[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/spacetime_whitney/spacetime_whitney.ipynb)

# Spacetime Whitney Waves

Spacetime Whitney waves propagate a scalar pulse through a triangulated worldsheet. An advancing mirror compresses the reflected pulse and supplies its extra energy; the mesh follows the moving boundary while spacetime stays flat.

The signed triangle patch defines Whitney interpolation independently of the metric. The signature enters through `Plane.dual()`, and its pullback by the gradient gives the wave operator. Equal space and time units make the light rays diagonal in every worldsheet view.


In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, stack
from numga.sparse import SparseExtensor, spdiag
from examples import instantiate
from examples.relativity.spacetime_whitney import render, scenarios
from examples.util.animation import save_figure, show_animation

mesh_module = instantiate("examples.util.mesh", Algebra("x+t-"))
ga, Mesh, as_scalar, mv = mesh_module.ga, mesh_module.Mesh, mesh_module.as_scalar, mesh_module.mv
Scalar = ga.gatype.scalar()                                                  # Scalar+
Plane = ga.gatype.vector()                                                   # Vector+
Bivector = ga.gatype.bivector()                                              # Bivector+
Point = ga.gatype.antivector()                                               # Antivector-
Patch = ga.gatype.antibivector()                                             # Antibivector-
Amount = ga.gatype.antiscalar()                                              # Antiscalar-

space, time = mv.x.dual(), -mv.t.dual()                                      # [] Point-, Point-

np.set_printoptions(precision=4, suppress=True)

## Whitney forms and the light cone

A triangle's corner hats have the same affine shape in Euclidean and Minkowski geometry. The Lorentzian dual changes which directions carry a wave. Initial and boundary data then select the causal solution: here the cavity is quiet before a localized source emits a pulse in each spatial direction.


In [ ]:
n_triangle_subdivisions = 7
n_wake_space = 81
n_wake_time = 140
wake_duration = 0.78
source_event = 0.5 * space + 0.12 * time                                     # [] Point-
source_width = 0.032

In [ ]:
corners, points = scenarios.single_triangle(space, time, n_triangle_subdivisions)   # [corners] Point-, [samples] Point-
sides = corners[[2, 0, 1]] - corners[[1, 2, 0]]                              # [corners] Point-
patch = (sides[0] & sides[1]) / 2                                            # [] Patch-
gradients = sides / (2 * patch)                                              # [corners] Plane+
hats = 1 + (gradients[:, None] & (points[None, :] - corners[:, None]))       # [corners, samples] Scalar+
edge_forms = (points[None, :] - corners[:, None]) / (2 * patch)              # [corners, samples] Plane+

wake_mesh, wake_vertices, _ = scenarios.spacetime_strip(
    space, time, n_wake_space, n_wake_time, np.ones(n_wake_time), wake_duration,
)                                                                            # Mesh, Point-[vertices], float
wake_edges = wake_mesh.d[0] * wake_vertices                                  # Point-[edges]
wake_sides = wake_mesh.boundary[2, 1] * wake_edges                           # [corners] Point-[faces]
wake_patches = (wake_sides[0] & wake_sides[1]) / 2                           # Patch-[faces]
wake_areas = wake_patches.norm()                                             # Scalar+[faces]
wake_corner_gradients = wake_sides / (2 * wake_patches)                      # [corners] Plane+[faces]
wake_gradient = (spdiag(wake_corner_gradients) * wake_mesh.incidence[2, 0]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[faces <- vertices]
wake_wave_operator = wake_gradient.adjugate()(spdiag(wake_areas * Plane.dual())(wake_gradient))   # (Amount- <- Scalar+)[vertices <- vertices]

# A time-derivative Gaussian source radiates two sharp pulses along the forward 45-degree null rays.
displacements = wake_vertices.batch() - source_event                         # [vertices] Point-
delay = mv.t & displacements                                                 # [vertices] Scalar+
distance_squared = (mv.x & displacements).squared() + delay.squared()        # [vertices] Scalar+
profile = (delay / source_width) * (-(distance_squared / (2 * source_width**2))).exp()   # [vertices] Scalar+
source = (profile * wake_areas.sites.mean()).field().dual()                  # Amount-[vertices]

wake_grid = np.arange(n_wake_time * n_wake_space).reshape(n_wake_time, n_wake_space)
wake_interior = ~SparseExtensor.selection(mv.context, wake_grid[1:-1, 1:-1].ravel(), wake_mesh.n_vertices) * Scalar   # (Scalar+ <- Scalar+)[vertices <- interior]
wake_future = ~SparseExtensor.selection(mv.context, wake_grid[2:, 1:-1].ravel(), wake_mesh.n_vertices) * Scalar   # (Scalar+ <- Scalar+)[vertices <- future]
wake = wake_future(wake_interior.adjugate()(wake_wave_operator(wake_future)).solve(wake_interior.adjugate()(source)))   # Scalar+[vertices]

save_figure(
    render.draw_light_cone_whitney(
        corners, points, gradients, edge_forms,
        wake_mesh, wake_vertices, wake, source_event,
    ),
    "spacetime_whitney_cone",
);

The first panel shows a corner hat's level sets and two directions from their common centre: its Euclidean normal in blue and its Lorentzian dual in orange. The edge form's dual follows the hyperbolic curves in the second panel. The wake in the third panel follows the dashed null rays and reflects off the walls; its arrow of time comes from the quiet initial rows.

In finite element exterior calculus, the scalar action is $\frac{1}{2}\int d\phi\wedge\star d\phi$, with discrete wave operator $K_{pq}=\int d\lambda_p\wedge\star d\lambda_q$.


In [ ]:
print("Spacetime triangle area:", patch.norm().to_array())
print("Corner gradient Lorentzian squares:", (gradients & Plane.dual()(gradients)).to_array())

## Moving-mirror Doppler compression

The advancing mirror forces the pulse into a shorter interval before sending it back. The scalar potential stays zero on both mirror worldlines; the reflected packet has a shorter wavelength and a larger time derivative. The mirror's prescribed motion supplies the energy gain.


In [ ]:
n_mirror_space = 85
n_mirror_time = 380
mirror_duration = 1.25
initial_length = 1.0
stroke = 0.36
inflection_time = 0.54
transition_time = 0.42
pulse_centre = 0.28
pulse_width = 0.085
wavelength = 0.17
before_row = 72
after_row = 310
n_frames = 48
frame_duration_ms = 55

In [ ]:
mirror_times = np.linspace(0.0, mirror_duration, n_mirror_time)
wall_positions = scenarios.mirror_trajectory(
    mirror_times, initial_length, stroke, inflection_time, transition_time,
)
mirror_mesh, mirror_vertices, dt = scenarios.spacetime_strip(
    space, time, n_mirror_space, n_mirror_time, wall_positions, mirror_duration,
)                                                                            # Mesh, Point-[vertices], float

edges = mirror_mesh.d[0] * mirror_vertices                                   # Point-[edges]
sides = mirror_mesh.boundary[2, 1] * edges                                   # [corners] Point-[faces]
patches = (sides[0] & sides[1]) / 2                                          # Patch-[faces]
areas = patches.norm()                                                       # Scalar+[faces]
corner_gradients = sides / (2 * patches)                                     # [corners] Plane+[faces]
gradient = (spdiag(corner_gradients) * mirror_mesh.incidence[2, 0]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[faces <- vertices]

# The Lorentzian dual is the only metric-dependent part of the wave operator.
wave_operator = gradient.adjugate()(spdiag(areas * Plane.dual())(gradient))  # (Amount- <- Scalar+)[vertices <- vertices]

# Pairing with the null plane `mv.x - mv.t` reads the phase of the right-moving wave at every event.
phase = ((mv.x - mv.t) & mirror_vertices).batch() - pulse_centre             # [vertices] Scalar+
envelope = (-(phase / pulse_width).squared()).exp()                          # [vertices] Scalar+
carrier = (2 * np.pi * phase / wavelength).cos()                             # [vertices] Scalar+

grid = np.arange(n_mirror_time * n_mirror_space).reshape(n_mirror_time, n_mirror_space)
seed = SparseExtensor.selection(mv.context, grid[:2, 1:-1].ravel(), mirror_mesh.n_vertices)   # Scalar+[seed <- vertices]
initial = ~seed * (seed * (envelope * carrier).field())                      # Scalar+[vertices]

# Initial rows carry the incident pulse; the omitted wall values stay zero.
interior = ~SparseExtensor.selection(mv.context, grid[1:-1, 1:-1].ravel(), mirror_mesh.n_vertices) * Scalar   # (Scalar+ <- Scalar+)[vertices <- interior]
future = ~SparseExtensor.selection(mv.context, grid[2:, 1:-1].ravel(), mirror_mesh.n_vertices) * Scalar   # (Scalar+ <- Scalar+)[vertices <- future]
rhs = -interior.adjugate()(wave_operator(initial))                           # Amount-[interior]
potential = initial + future(interior.adjugate()(wave_operator(future)).solve(rhs))   # Scalar+[vertices]

field = gradient(potential)                                                  # Plane+[faces]
face_rate = field & time                                                     # Scalar+[faces]
to_vertices = ~mirror_mesh.incidence[2, 0].sum(axis=0)                       # Scalar+[vertices <- faces]
wave_rate = (to_vertices * (areas * face_rate)) / (to_vertices * areas)      # Scalar+[vertices]

# Reflecting the Lorentzian dual across the observer's slice mv.t flips the temporal sign,
# turning the Lagrangian density into the positive-definite Hamiltonian wave energy density.
energy_density = -(field & (mv.t >> Plane.dual()(field))) / 2                # Scalar+[faces]
slab_indices = mirror_mesh.faces.min(axis=-1) // n_mirror_space
to_slabs = SparseExtensor.from_indices(
    as_scalar(np.ones(len(mirror_mesh.faces)) / dt),
    slab_indices, np.arange(len(mirror_mesh.faces)), (n_mirror_time - 1, len(mirror_mesh.faces)),
)                                                                            # Scalar+[layers <- faces]
slice_energies = (to_slabs * (areas * energy_density)).batch()               # [layers] Scalar+

save_figure(
    render.draw_moving_mirror_doppler(
        mirror_mesh, mirror_vertices, n_mirror_space, n_mirror_time,
        wave_rate, slice_energies, before_row, after_row,
    ),
    "spacetime_whitney_doppler",
);

The worldsheet resolves the incident and reflected light rays at equal space and time scale. Its dashed slices select the broad incident pulse in blue and the narrow reflected pulse in orange; the middle panel plots the scalar potential's time derivative relative to its incident peak, and the right panel shows the energy transferred during reflection.

In special-relativistic wave notation, a mirror advancing at constant speed $v$ shortens the wavelength and multiplies both $\partial_t\phi$ and the integrated pulse energy by $(1+v)/(1-v)$. Here the speed changes during reflection, so the pulse samples a range of Doppler factors.


In [ ]:
wave_rate_grid = wave_rate.batch().reshape(n_mirror_time, n_mirror_space)     # [times, space] Scalar+
before_norms = wave_rate_grid[before_row].norm()                              # [space] Scalar+
after_norms = wave_rate_grid[after_row].norm()                                # [space] Scalar+
print("Peak wave-rate ratio (after / before):", (after_norms[after_norms.argmax()] / before_norms[before_norms.argmax()]).to_array())
print("Wave energy ratio         (after / before):", (slice_energies[after_row] / slice_energies[before_row]).to_array())

## A spatial slice through the worldsheet

The horizontal slice links the spacetime history to the moving cavity. The mirror catches the incident pulse and releases a narrower, faster-oscillating packet back across the cavity.


In [ ]:
frame_rows = np.linspace(4, n_mirror_time - 4, n_frames, dtype=int)
show_animation(
    render.animate_moving_mirror(
        mirror_mesh, mirror_vertices, n_mirror_space, n_mirror_time,
        wave_rate, frame_rows,
    ),
    "spacetime_whitney_mirror",
    frame_duration_ms,
)

In [ ]:
# checks
corner_hats = 1 + (gradients[:, None] & (corners[None, :] - corners[:, None]))
np.testing.assert_allclose(corner_hats.kernel, np.eye(3)[..., None], atol=1e-14)
np.testing.assert_allclose(hats.sum(axis=0).kernel, 1.0, atol=1e-14)
np.testing.assert_allclose(gradients.sum(axis=0).kernel, 0.0, atol=1e-14)

first_idx, second_idx = [1, 2, 0], [2, 0, 1]
classic_forms = hats[first_idx] * gradients[second_idx, None] - hats[second_idx] * gradients[first_idx, None]
np.testing.assert_allclose((edge_forms - classic_forms).kernel, 0.0, atol=1e-14)
triangle_sides = corners[second_idx] - corners[first_idx]
midpoints = (corners[first_idx] + corners[second_idx]) / 2
midpoint_forms = (midpoints[None, :] - corners[:, None]) / (2 * patch)
np.testing.assert_allclose((midpoint_forms & triangle_sides[None, :]).kernel, np.eye(3)[..., None], atol=1e-14)

right_null, left_null = mv.x - mv.t, mv.x + mv.t
np.testing.assert_allclose((right_null & Plane.dual()(right_null)).kernel, 0.0, atol=1e-14)
np.testing.assert_allclose((left_null & Plane.dual()(left_null)).kernel, 0.0, atol=1e-14)

# The edge-potential Maxwell operator is checked here; the demonstration evolves the scalar wave above.
d0 = mirror_mesh.d[0] * Scalar                                               # (Scalar+ <- Scalar+)[edges <- vertices]
d1 = mirror_mesh.d[1] * Scalar                                               # (Scalar+ <- Scalar+)[faces <- edges]
centroid_forms = (corner_gradients[[2, 0, 1]] - corner_gradients[[1, 2, 0]]) / 3   # [corners] Plane+[faces]
whitney1 = (spdiag(centroid_forms) * mirror_mesh.boundary[2, 1]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[faces <- edges]
first, second = corner_gradients[[1, 2, 0]], corner_gradients[[2, 0, 1]]     # [corners] Plane+[faces], [corners] Plane+[faces]
curl = (spdiag(2 * (first ^ second)) * mirror_mesh.boundary[2, 1]).sum(axis=0) * Scalar  # (Bivector+ <- Scalar+)[faces <- edges]
whitney2 = spdiag(patches.inverse().dual() * Scalar)                         # (Bivector+ <- Scalar+)[faces <- faces]
maxwell_operator = curl.adjugate()(spdiag(areas * Bivector.dual())(curl))    # (Amount- <- Scalar+)[edges <- edges]

test_phi = (mv.x & mirror_vertices) * 0.8 - (mv.t & mirror_vertices) * 1.2
np.testing.assert_allclose(d1(d0(test_phi)).kernel, 0.0, atol=1e-14)
np.testing.assert_allclose((whitney1(d0(test_phi)) - gradient(test_phi)).kernel, 0.0, atol=1e-12)
test_cochain = d0(test_phi) + as_scalar(np.sin(np.arange(len(mirror_mesh.edges)))).field()
np.testing.assert_allclose((curl(test_cochain) - whitney2(d1(test_cochain))).kernel, 0.0, atol=1e-9)
np.testing.assert_allclose(maxwell_operator(d0(test_phi)).kernel, 0.0, atol=1e-9)

outside_cone = ((mv.x & displacements).norm() > delay + 0.14) | (delay < -0.10)
wake_batch = wake.batch()
assert np.max(np.abs(wake_batch[outside_cone].kernel)) < 0.02 * np.max(np.abs(wake_batch.kernel))
assert np.max(np.abs(wave_rate_grid[after_row].kernel)) > 1.85 * np.max(np.abs(wave_rate_grid[before_row].kernel))
assert slice_energies[after_row] > 2.15 * slice_energies[before_row]